# 🔏 CausaGanha — Privacy Filter: Dispositivo Span Detection

Fine-tunes a Portuguese BERT model to detect the *dispositivo* section
in Brazilian judicial decisions as a token-classification (NER) task.

**No embeddings needed — runs on a free T4 in ~10 min.**

**Steps:**
1. Install dependencies
2. Download `textos.parquet` from Internet Archive
3. Generate JSONL dataset via `prepare_privacy_filter_dataset.py`
4. Convert spans → BIO token labels
5. Fine-tune `neuralmind/bert-base-portuguese-cased`
6. Evaluate & save model


## 1. 🔧 Install Dependencies

In [ ]:
!pip install -q transformers datasets seqeval ibis-framework[duckdb] structlog
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('✅ Dependencies installed')


## 2. 📥 Clone Repo & Download Data


In [ ]:
import os, urllib.request

REPO_URL = 'https://github.com/franklinbaldo/causaganha.git'
BRANCH = 'feat/embedder-smart-truncate-and-privacy-dataset-v2'
REPO_DIR = '/content/causaganha'

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
os.chdir(REPO_DIR)

# Download textos.parquet from Internet Archive
PARQUET_DIR = f'{REPO_DIR}/data/test_parquets'
os.makedirs(PARQUET_DIR, exist_ok=True)
TEXTOS_URL = 'https://archive.org/download/causaganha-test-parquets/textos.parquet'
TEXTOS_PATH = f'{PARQUET_DIR}/textos.parquet'

if not os.path.exists(TEXTOS_PATH):
    print('⬇️  Downloading textos.parquet from Internet Archive...')
    urllib.request.urlretrieve(TEXTOS_URL, TEXTOS_PATH)
    print(f'✅ Downloaded ({os.path.getsize(TEXTOS_PATH):,} bytes)')
else:
    print(f'⏭️  Already exists: {TEXTOS_PATH}')


## 3. 🏷️ Generate Dispositivo Span Dataset

Runs `prepare_privacy_filter_dataset.py` — pure regex, no GPU.
Outputs `data/privacy_filter/{train,validation,test}.jsonl`.


In [ ]:
import sys
sys.path.insert(0, f'{REPO_DIR}/src')

# Install the package so KeywordClassifier is importable
!pip install -q -e {REPO_DIR} --no-deps

!python {REPO_DIR}/scripts/prepare_privacy_filter_dataset.py

import os
for split in ['train', 'validation', 'test']:
    path = f'{REPO_DIR}/data/privacy_filter/{split}.jsonl'
    if os.path.exists(path):
        lines = sum(1 for _ in open(path))
        print(f'  ✅ {split}.jsonl — {lines} examples')
    else:
        print(f'  ❌ {split}.jsonl MISSING')


## 4. 🔄 Convert Spans → BIO Token Labels

Converts character-level `{"text": ..., "spans": {"dispositivo": [[start, end]]}}` to token-level BIO NER format for BERT fine-tuning.


In [ ]:
import json
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict

MODEL_NAME = 'neuralmind/bert-base-portuguese-cased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

LABEL2ID = {'O': 0, 'B-DISPOSITIVO': 1, 'I-DISPOSITIVO': 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f]

def char_spans_to_bio(example):
    text = example['text']
    spans = example.get('spans', {}).get('dispositivo', [])

    # Build a character-level label array
    char_labels = ['O'] * len(text)
    for start, end in spans:
        for i in range(start, min(end, len(text))):
            char_labels[i] = 'B-DISPOSITIVO' if i == start else 'I-DISPOSITIVO'

    # Tokenize and align labels to sub-word tokens
    enc = tokenizer(text, truncation=True, max_length=512,
                    return_offsets_mapping=True)
    token_labels = []
    for start, end in enc['offset_mapping']:
        if start == end:  # special token
            token_labels.append(-100)
        else:
            # Use the label of the first character of this token
            token_labels.append(LABEL2ID[char_labels[start]])

    enc['labels'] = token_labels
    enc.pop('offset_mapping')
    return enc

splits = {}
for split in ['train', 'validation', 'test']:
    path = f'{REPO_DIR}/data/privacy_filter/{split}.jsonl'
    raw = load_jsonl(path)
    ds = Dataset.from_list(raw)
    splits[split] = ds.map(char_spans_to_bio, remove_columns=ds.column_names)
    print(f'  {split}: {len(splits[split])} examples tokenized')

dataset = DatasetDict(splits)
print('\n✅ Dataset ready:', dataset)


## 5. 🏋️ Fine-tune BERTimbau

In [ ]:
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
import numpy as np
from seqeval.metrics import classification_report as seq_report

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

def compute_metrics(p):
    preds, labels = p
    preds = np.argmax(preds, axis=2)
    true_seqs, pred_seqs = [], []
    for pred_row, label_row in zip(preds, labels):
        true_seq, pred_seq = [], []
        for p_id, l_id in zip(pred_row, label_row):
            if l_id == -100:
                continue
            true_seq.append(ID2LABEL[l_id])
            pred_seq.append(ID2LABEL[p_id])
        true_seqs.append(true_seq)
        pred_seqs.append(pred_seq)
    report = seq_report(true_seqs, pred_seqs, output_dict=True, zero_division=0)
    return {
        'precision': report['DISPOSITIVO']['precision'],
        'recall': report['DISPOSITIVO']['recall'],
        'f1': report['DISPOSITIVO']['f1-score'],
    }

args = TrainingArguments(
    output_dir='/content/privacy-filter-dispositivo',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=20,
    fp16=True,
    report_to='none',
)

collator = DataCollatorForTokenClassification(tokenizer)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

print('🏋️  Training...')
trainer.train()
print('✅ Training complete')


## 6. ✅ Evaluate on Test Set

In [ ]:
results = trainer.evaluate(dataset['test'])
print('\n📊 Test set results:')
print(f"  Precision : {results['eval_precision']:.4f}")
print(f"  Recall    : {results['eval_recall']:.4f}")
print(f"  F1        : {results['eval_f1']:.4f}")


## 7. 📦 Save & Download Model

In [ ]:
import shutil
from google.colab import files

SAVE_DIR = '/content/privacy-filter-dispositivo-final'
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'✅ Model saved to {SAVE_DIR}')

# Zip and download
zip_path = '/content/privacy_filter_dispositivo.zip'
shutil.make_archive('/content/privacy_filter_dispositivo', 'zip', SAVE_DIR)
files.download(zip_path)
print('📦 Downloaded privacy_filter_dispositivo.zip')

# Show how to use it
print('''
To use locally:
  from transformers import pipeline
  ner = pipeline('token-classification', model=SAVE_DIR, aggregation_strategy='simple')
  ner("DISPOSITIVO: Julgo procedente o pedido.")
''')
